# 05 - Train Baseline

Name: Usha Oudeacoumar
Student ID: 20057009
Email: 20057009@mydbs.ie

This notebook trains the two baseline models using only the development years data
(2011-2014). First it splits the development data into a train part and a small
validation part (to check the model later in notebook 06). It drops any feature
column that has basically no variation (not useful for the model). Then it trains
a Logistic Regression and a Gradient Boosting model, does a small grid search to
pick ok-ish hyperparameters, and saves the trained models so later notebooks can
load them without retraining.

In [6]:
import sys
sys.path.append(".")
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from utils import PROC_DIR, MODELS_DIR, load_json

cfg = load_json("../configs/study_config.json")
dev_years = cfg["dev_years"]
eval_years = cfg["eval_years"]

data = pd.read_csv(os.path.join(PROC_DIR, "model_ready.csv"))
feat_cols = [c for c in data.columns if c not in ("yr", "admit")]
data.shape

(160526, 92)

In [7]:
# drop columns that barely change at all, they are not going to help the model
var_check = data[feat_cols].var()
low_var_cols = var_check[var_check < 0.001].index.tolist()
print("dropping low variance cols:", low_var_cols)

feat_cols = [c for c in feat_cols if c not in low_var_cols]
len(feat_cols)

dropping low variance cols: ['cat__rfv_group_21', 'cat__rfv_group_24', 'cat__rfv_group_29', 'cat__rfv_group_35', 'cat__rfv_group_44', 'cat__rfv_group_62', 'cat__rfv_group_63', 'cat__rfv_group_64', 'cat__rfv_group_65', 'cat__rfv_group_66']


80

In [8]:
# split off the development years, then split that into train and a small
# validation set for notebook 06 to check the model on before the real test
dev_df = data[data["yr"].isin(dev_years)].reset_index(drop=True)
eval_df = data[data["yr"].isin(eval_years)].reset_index(drop=True)

train_df, val_df = train_test_split(
    dev_df, test_size=0.2, random_state=42, stratify=dev_df["admit"]
)

print("train:", train_df.shape, "val:", val_df.shape, "eval (future years):", eval_df.shape)

train_df.to_csv(os.path.join(PROC_DIR, "dev_train.csv"), index=False)
val_df.to_csv(os.path.join(PROC_DIR, "dev_val.csv"), index=False)
eval_df.to_csv(os.path.join(PROC_DIR, "eval_data.csv"), index=False)

train: (68297, 92) val: (17075, 92) eval (future years): (75154, 92)


In [9]:
# train model A - logistic regression. NOT using class_weight balanced on purpose,
# this whole project is about checking if predicted probabilities are calibrated,
# and balancing classes would change the predicted probabilities on purpose which
# would mess up calibration from the start
X_train, y_train = train_df[feat_cols], train_df["admit"]

log_reg_grid = {"C": [0.01, 0.1, 1, 10]}
log_reg = GridSearchCV(
    LogisticRegression(max_iter=5000),
    log_reg_grid, scoring="roc_auc", cv=3,
)
log_reg.fit(X_train, y_train)
print("best C:", log_reg.best_params_)

best C: {'C': 1}


In [10]:
# train model B - gradient boosting, small grid search too
gb_grid = {"max_depth": [3, 5], "max_iter": [100, 200]}
gb = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    gb_grid, scoring="roc_auc", cv=3,
)
gb.fit(X_train, y_train)
print("best params:", gb.best_params_)

best params: {'max_depth': 3, 'max_iter': 200}


In [11]:
# save the frozen models plus the feature list used, so notebook 06 onwards can
# load them without retraining (these should not be touched again from here on)
joblib.dump(log_reg.best_estimator_, os.path.join(MODELS_DIR, "log_reg_model.pkl"))
joblib.dump(gb.best_estimator_, os.path.join(MODELS_DIR, "gb_model.pkl"))
joblib.dump(feat_cols, os.path.join(MODELS_DIR, "feat_cols.pkl"))
print("saved both baseline models")

saved both baseline models
